# Análise de Desempenho: PostgreSQL (JSONB) vs MongoDB

## 1. Configuração do Ambiente

Carrega as variáveis de conexão a partir dos arquivos `.env` de cada implementação.

In [ ]:
import os
import json
import uuid
import random
from faker import Faker
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from pymongo import MongoClient
from bson import ObjectId
import pandas as pd

pd.set_option('display.max_colwidth', None)

# Carrega as variáveis de ambiente dos dois projetos
load_dotenv("../catalogo-postgres/.env")
load_dotenv("../catalogo-mongo/.env")

# Conexão PostgreSQL (via SQLAlchemy)
pg_user = os.getenv("PGUSER")
pg_password = os.getenv("PGPASSWORD")
pg_db = os.getenv("PGDATABASE")
pg_host = os.getenv("PGHOST")
pg_port = os.getenv("PGPORT")

engine_pg = create_engine(f"postgresql://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_db}")

# Conexão MongoDB (via PyMongo)
mongo_uri = os.getenv("MONGO_URI")
mongo_db_name = os.getenv("MONGO_DB")

client_mongo = MongoClient(mongo_uri)
db_mongo = client_mongo[mongo_db_name]

random.seed(42)  # gera as mesmas coisas todas as vezes
fake = Faker("pt_BR")
Faker.seed(42)

## 2. Estrutura do Banco (Schema)

Cria as tabelas do PostgreSQL (`restaurantes`, `itens`) com as constraints definidas na modelagem (chave primária nomeada, chave estrangeira com `ON DELETE CASCADE`). 

No MongoDB, as coleções não exigem criação explícita — apenas são limpas aqui para permitir uma reexecução limpa do notebook.

In [16]:
RESETAR_DADOS = True  # mude para False para preservar dados já existentes

DDL_POSTGRES = """
DROP TABLE IF EXISTS itens CASCADE;
DROP TABLE IF EXISTS restaurantes CASCADE;

CREATE TABLE restaurantes (
  id          UUID DEFAULT gen_random_uuid(),
  nome        VARCHAR(255) NOT NULL,
  categoria   VARCHAR(100),
  ativo       BOOLEAN NOT NULL DEFAULT true,
  criado_em   TIMESTAMP NOT NULL DEFAULT now(),

  CONSTRAINT pk_restaurante PRIMARY KEY (id)
);

CREATE TABLE itens (
  id                    UUID DEFAULT gen_random_uuid(),
  restaurante_id        UUID NOT NULL,
  nome                  VARCHAR(255) NOT NULL,
  categoria             VARCHAR(100) NOT NULL,
  disponivel            BOOLEAN NOT NULL DEFAULT true,
  preco                 NUMERIC(10,2) NOT NULL,
  atributos_variaveis   JSONB NOT NULL DEFAULT '{}'::jsonb,
  criado_em             TIMESTAMP NOT NULL DEFAULT now(),
  atualizado_em         TIMESTAMP NOT NULL DEFAULT now(),

  CONSTRAINT pk_item PRIMARY KEY (id),
  CONSTRAINT fk_item_restaurante FOREIGN KEY (restaurante_id)
    REFERENCES restaurantes(id) ON DELETE CASCADE
);
"""

if RESETAR_DADOS:
    with engine_pg.begin() as conn:
        conn.execute(text(DDL_POSTGRES))
    print("Tabelas do PostgreSQL recriadas.")

    db_mongo.restaurantes.drop()
    db_mongo.itens.drop()
    print("Coleções do MongoDB limpas.")
else:
    print("RESETAR_DADOS = False — estrutura/dados existentes preservados.")

Tabelas do PostgreSQL recriadas.
Coleções do MongoDB limpas.


## 3. Geração de Dados

Gera restaurantes e itens de forma sintética, com os atributos variáveis mudando conforme a categoria do item (`pizza`, `bebida`, `sobremesa`, `lanche`) — reproduzindo a heterogeneidade real de um catálogo, conforme a modelagem de dados do TCC. 

Os mesmos dados são usados nos dois bancos; apenas o formato do identificador muda (`UUID` no PostgreSQL, `ObjectId` no MongoDB).

In [17]:
NUM_RESTAURANTES = 1000
NUM_ITENS = 1000000

CATEGORIAS_REST = ['italiana', 'brasileira', 'árabe', 'mexicana']
CATEGORIAS_ITEM = ['pizza', 'bebida', 'sobremesa', 'lanche']

PREFIXOS_POR_CATEGORIA_REST = {
    'italiana': ['Cantina', 'Trattoria', 'Pizzaria'],
    'brasileira': ['Sabor', 'Empório', 'Fogão de Chão'],
    'árabe': ['Oásis', 'Empório Árabe', 'Casa Árabe'],
    'mexicana': ['Cantina Mexicana', 'Casa do Taco', 'Sabor Mexicano'],
}

NOMES_POR_CATEGORIA = {
    'pizza': ['Margherita', 'Calabresa', 'Quatro Queijos', 'Frango com Catupiry', 'Portuguesa',
              'Napolitana', 'Vegetariana', 'Pepperoni'],
    'bebida': ['Coca-Cola Lata', 'Guaraná Antarctica', 'Suco de Laranja', 'Água Mineral',
               'Suco de Uva', 'Chá Gelado', 'Água com Gás'],
    'sobremesa': ['Petit Gateau', 'Pudim', 'Mousse de Maracujá', 'Brownie',
                  'Sorvete', 'Torta de Limão', 'Cheesecake'],
    'lanche': ['X-Burger', 'X-Salada', 'X-Bacon', 'X-Tudo', 'X-Frango', 'Wrap de Frango'],
}

COMPLEMENTOS_ITEM = ['Tradicional', 'Especial', 'da Casa', 'Artesanal', 'Premium', 'Clássico', None]

TAGS = ['vegetariano', 'vegano', 'mais vendido', 'sem glúten', 'sem lactose', 'picante', 'novidade']
INGREDIENTES = ['queijo', 'bacon', 'alface', 'tomate', 'ovo', 'molho especial',
                 'cebola caramelizada', 'picles', 'maionese temperada', 'cheddar']

def sortear_tags():
    return [t for t in TAGS if random.random() > 0.7]

def sortear_ingredientes():
    escolhidos = [i for i in INGREDIENTES if random.random() > 0.5]
    return escolhidos if len(escolhidos) >= 2 else INGREDIENTES[:2]

def gerar_promocao():
    return {"descontoPercentual": random.randint(5, 30), "validoAte": "2026-12-31"}

def gerar_atributos(categoria):
    if categoria == 'pizza':
        attrs = {"opcoes": [{"nome": "Borda recheada", "precoAdicional": 8.0}],
                 "tamanhos": ["média", "grande", "família"], "tags": sortear_tags()}
        if random.random() > 0.6:
            attrs["promocao"] = gerar_promocao()
        return attrs
    if categoria == 'bebida':
        return {"volumeMl": random.choice([350, 500, 600]),
                "gelada": random.random() > 0.2, "tags": sortear_tags()}
    if categoria == 'sobremesa':
        attrs = {"informacaoNutricional": {"calorias": random.randint(200, 500),
                                             "contemGluten": random.random() > 0.5}}
        if random.random() > 0.7:
            attrs["promocao"] = gerar_promocao()
        return attrs
    attrs = {"ingredientes": sortear_ingredientes(),
             "tamanho": random.choice(["único", "médio", "grande"]),
             "opcoes": [{"nome": "Sem cebola", "precoAdicional": 0}],
             "tags": sortear_tags()}
    if random.random() > 0.65:
        attrs["promocao"] = gerar_promocao()
    return attrs

def gerar_preco(categoria):
    faixas = {'pizza': (35, 70), 'bebida': (5, 15), 'sobremesa': (12, 25), 'lanche': (18, 35)}
    lo, hi = faixas[categoria]
    return round(random.uniform(lo, hi), 2)

def gerar_nome_restaurante(categoria, usados):
    prefixo = random.choice(PREFIXOS_POR_CATEGORIA_REST[categoria])
    for _ in range(10):
        complemento = random.choice([fake.last_name(), fake.city()])
        nome = f"{prefixo} {complemento}"
        if nome not in usados:
            return nome
    return f"{prefixo} {fake.unique.last_name()}"  # fallback raro para garantir unicidade

def gerar_nome_item(categoria):
    base = random.choice(NOMES_POR_CATEGORIA[categoria])
    complemento = random.choice(COMPLEMENTOS_ITEM)
    return f"{base} {complemento}" if complemento else base

restaurantes = []
nomes_restaurantes_usados = set()
for i in range(NUM_RESTAURANTES):
    categoria = CATEGORIAS_REST[i % len(CATEGORIAS_REST)]
    nome = gerar_nome_restaurante(categoria, nomes_restaurantes_usados)
    nomes_restaurantes_usados.add(nome)
    restaurantes.append({
        "pg_id": str(uuid.uuid4()),
        "mongo_id": ObjectId(),
        "nome": nome,
        "categoria": categoria,
        "ativo": True,
    })

itens = []
for i in range(NUM_ITENS):
    restaurante = restaurantes[i % NUM_RESTAURANTES]
    categoria = CATEGORIAS_ITEM[i % len(CATEGORIAS_ITEM)]
    itens.append({
        "restaurante": restaurante,
        "nome": gerar_nome_item(categoria),
        "categoria": categoria,
        "disponivel": random.random() > 0.1,
        "preco": gerar_preco(categoria),
        "atributos": gerar_atributos(categoria),
    })

print(f"Gerados: {len(restaurantes)} restaurantes e {len(itens)} itens.")

Gerados: 1000 restaurantes e 1000000 itens.


## 4. Inserção dos Dados

### PostgreSQL

In [18]:
with engine_pg.begin() as conn:
    conn.execute(
        text("""
            INSERT INTO restaurantes (id, nome, categoria, ativo)
            VALUES (:id, :nome, :categoria, :ativo)
        """),
        [
            {"id": r["pg_id"], "nome": r["nome"], "categoria": r["categoria"], "ativo": r["ativo"]}
            for r in restaurantes
        ]
    )

    conn.execute(
        text("""
            INSERT INTO itens (restaurante_id, nome, categoria, disponivel, preco, atributos_variaveis)
            VALUES (:restaurante_id, :nome, :categoria, :disponivel, :preco, CAST(:atributos AS JSONB))
        """),
        [
            {
                "restaurante_id": it["restaurante"]["pg_id"],
                "nome": it["nome"],
                "categoria": it["categoria"],
                "disponivel": it["disponivel"],
                "preco": it["preco"],
                "atributos": json.dumps(it["atributos"], ensure_ascii=False),
            }
            for it in itens
        ]
    )

print(f"PostgreSQL: {len(restaurantes)} restaurantes e {len(itens)} itens inseridos.")

PostgreSQL: 1000 restaurantes e 1000000 itens inseridos.


### MongoDB

In [19]:
docs_restaurantes = [
    {"_id": r["mongo_id"], "nome": r["nome"], "categoria": r["categoria"], "ativo": r["ativo"]}
    for r in restaurantes
]
db_mongo.restaurantes.insert_many(docs_restaurantes)

docs_itens = []
for it in itens:
    doc = {
        "restauranteId": it["restaurante"]["mongo_id"],
        "nome": it["nome"],
        "categoria": it["categoria"],
        "disponivel": it["disponivel"],
        "preco": it["preco"],
    }
    doc.update(it["atributos"])  # campos variáveis direto no documento
    docs_itens.append(doc)

db_mongo.itens.insert_many(docs_itens)

print(f"MongoDB: {len(docs_restaurantes)} restaurantes e {len(docs_itens)} itens inseridos.")

MongoDB: 1000 restaurantes e 1000000 itens inseridos.


## 5. Funções de Benchmark

Define uma função genérica de cronometragem, reutilizável tanto para consultas no PostgreSQL quanto no MongoDB. 

A função executa uma ação repetidas vezes (com aquecimento prévio de cache), retornando estatísticas de tempo (`mínimo`, `máximo`, `média`, `mediana`) para cada cenário testado.

- `acao` — a tarefa que você quer cronometrar
- `repeats` — quantas vezes repetir a medição (padrão: 5)
- `warm_up` — se deve fazer uma execução de "aquecimento" antes de começar a cronometrar (padrão: sim)

In [ ]:
import statistics

def run_benchmark(acao, repeats=5, warm_up=True):
    try:
        if warm_up:
            acao()  # descarta a primeira execução

        tempos = []
        for _ in range(repeats):
            tempo_reportado = acao() # executa a consulta
            tempos.append(tempo_reportado)

        return {
            "min": min(tempos),
            "max": max(tempos),
            "media": statistics.mean(tempos),
            "mediana": statistics.median(tempos),
            "tempos_brutos": tempos,
        }
    except Exception as e:
        print("Erro ao executar benchmark:", e)
        return None

## 6. Consultas sem Índice

### PostgreSQL

#### Dados

In [ ]:
def consulta_pizzas_pg():
    return pd.read_sql(
        "SELECT nome, categoria, preco FROM itens WHERE categoria = 'pizza'",
        engine_pg
    )

df_pg = consulta_pizzas_pg()
print(f"PostgreSQL: {len(df_pg)} pizzas encontradas")
display(df_pg.head())

PostgreSQL: 250000 pizzas encontradas


,nome,categoria,preco
0,Portuguesa Especial,pizza,60.38
1,Frango com Catupiry,pizza,65.01
2,Frango com Catupiry da Casa,pizza,36.60
3,Portuguesa Premium,pizza,35.60
4,Margherita Especial,pizza,61.33


In [39]:
# 2. Ver o plano + servir de base para o benchmark
def explain_pizzas_pg():
    with engine_pg.connect() as conn:
        resultado = conn.execute(text(
            "EXPLAIN (ANALYZE, FORMAT JSON) SELECT nome, categoria, preco FROM itens WHERE categoria = 'pizza'"
        )).scalar()
        return resultado[0]

plano_pg = explain_pizzas_pg()
print(plano_pg["Plan"])

{'Node Type': 'Seq Scan', 'Parallel Aware': False, 'Async Capable': False, 'Relation Name': 'itens', 'Alias': 'itens', 'Startup Cost': 0.0, 'Total Cost': 48333.12, 'Plan Rows': 244216, 'Plan Width': 33, 'Actual Startup Time': 0.44, 'Actual Total Time': 262.749, 'Actual Rows': 250000, 'Actual Loops': 1, 'Filter': "((categoria)::text = 'pizza'::text)", 'Rows Removed by Filter': 750000}


In [40]:
# 3. Medir o tempo, usando o cronômetro genérico (run_benchmark) + explain_pizzas_pg
resultado_pg = run_benchmark(lambda: explain_pizzas_pg()["Execution Time"], repeats=10)
print(resultado_pg)

{'min': 169.139, 'max': 235.96, 'media': 197.4728, 'mediana': 199.5045, 'tempos_brutos': [169.139, 173.565, 208.372, 235.96, 197.961, 201.048, 194.538, 206.064, 203.122, 184.959]}


### MongoDB

In [41]:
# 1. Ver os dados (não entra no benchmark, só conferência visual)
def consulta_pizzas_mongo():
    cursor = db_mongo.itens.find(
        {"categoria": "pizza"},
        {"_id": 0, "nome": 1, "categoria": 1, "preco": 1}
    )
    return pd.DataFrame(list(cursor))

df_mongo = consulta_pizzas_mongo()
print(f"MongoDB: {len(df_mongo)} pizzas encontradas")
display(df_mongo.head())

MongoDB: 250000 pizzas encontradas


,nome,categoria,preco
0,Portuguesa Especial,pizza,60.38
1,Frango com Catupiry,pizza,65.01
2,Frango com Catupiry da Casa,pizza,36.60
3,Portuguesa Premium,pizza,35.60
4,Margherita Especial,pizza,61.33


In [42]:
# 2. Ver o plano + servir de base para o benchmark
def explain_pizzas_mongo():
    return db_mongo.itens.find(
        {"categoria": "pizza"},
        {"_id": 0, "nome": 1, "categoria": 1, "preco": 1}
    ).explain()

plano_mongo = explain_pizzas_mongo()
stats = plano_mongo["executionStats"]

print(f"Stage: {plano_mongo['queryPlanner']['winningPlan'].get('stage')}")
print(f"Documentos examinados: {stats['totalDocsExamined']}")
print(f"Chaves de índice examinadas: {stats['totalKeysExamined']}")
print(f"Documentos retornados: {stats['nReturned']}")

Stage: PROJECTION_SIMPLE
Documentos examinados: 1000000
Chaves de índice examinadas: 0
Documentos retornados: 250000


In [43]:
# 3. Medir o tempo, usando o cronômetro genérico (run_benchmark) + explain_pizzas_mongo
resultado_mongo = run_benchmark(lambda: explain_pizzas_mongo()["executionStats"]["executionTimeMillis"], repeats=10)
print(resultado_mongo)

{'min': 505, 'max': 593, 'media': 526.5, 'mediana': 518.0, 'tempos_brutos': [513, 541, 509, 505, 593, 523, 511, 534, 512, 524]}


## 6. Consultas Índice GIN